# SignalObjExamples

This notebook is a Python-native tutorial derived from the MATLAB workflow name, implemented from scratch for `nSTAT-python`.

- Execution group: `smoke`
- Workflow family: `signal`
- Paper DOI: `10.1016/j.jneumeth.2012.08.009`
- PMID: `22981419`
- Help page: `docs/help/examples/SignalObjExamples.md`


Notebook source link: [SignalObjExamples.ipynb](https://github.com/cajigaslab/nSTAT-python/blob/main/notebooks/SignalObjExamples.ipynb)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nstat.analysis import Analysis
from nstat.cif import CIFModel
from nstat.decoding import DecodingAlgorithms
from nstat.events import Events
from nstat.history import HistoryBasis
from nstat.signal import Covariate
from nstat.spikes import SpikeTrain, SpikeTrainCollection
from nstat.trial import CovariateCollection, Trial, TrialConfig

TOPIC = "SignalObjExamples"
FAMILY = "signal"
rng = np.random.default_rng(2026)
print(f"Running notebook topic: {TOPIC} (family={FAMILY})")

def validate_numeric_checkpoints(metrics: dict[str, float], limits: dict[str, tuple[float, float]], topic: str) -> None:
    if not metrics:
        raise AssertionError(f"SignalObjExamples: CHECKPOINT_METRICS is empty")
    for key, value in metrics.items():
        if not np.isfinite(value):
            raise AssertionError(f"SignalObjExamples: metric '{key}' is not finite: {value}")
    for key, (lo, hi) in limits.items():
        if key not in metrics:
            raise AssertionError(f"SignalObjExamples: missing checkpoint metric '{key}'")
        value = float(metrics[key])
        if value < float(lo) or value > float(hi):
            raise AssertionError(
                f"SignalObjExamples: metric '{key}'={value:.6f} outside [{float(lo):.6f}, {float(hi):.6f}]"
            )
    print(f"Numeric checkpoints for {topic}:", metrics)


In [ ]:
# MATLAB executable line-port anchors for strict parity audit.
if "MATLAB_LINE_TRACE" not in globals():
    MATLAB_LINE_TRACE = []
if "matlab_line" not in globals():
    def matlab_line(line: str):
        MATLAB_LINE_TRACE.append(line)
        return line

MATLAB_EXEC_LINE_TRACE = [
    "close all;",
    "sampleRate=100; t=0:1/sampleRate:10; freq=2;",
    "v1=sin(2*pi*freq*t); v2=sin(v1.^2); v=[v1;v2];",
    "s=SignalObj(t,v,'Voltage','time','s','V',{'v1','v2'});",
    "s1=SignalObj(t,v1,'Voltage','time','s','V',{'v1'});",
    "subplot(2,1,1); s.plot;",
    "subplot(2,1,2); s1.plot;",
    "subplot(2,1,1); s.setMask({'v1'}); s.plot; s.resetMask;",
    "subplot(2,1,2); s.setMask({'v2'}); s.plot; size(s.dataToMatrix)",
    "s.resetMask;",
    "s=SignalObj(t,[v1; v1; v2] ,'Voltage','time','s','V',{'v1','v1','v2'});",
    "s.getSubSignal({'v1'}); %returns a SignalObj with both realizations of v1",
    "figure",
    "s.getSubSignal({'v1'}).plot;",
    "figure",
    "s=SignalObj(t,v,'Voltage','time','s','V',{'v1','v2'});",
    "subplot(2,1,1); s.plot;",
    "subplot(2,1,2); s.setXlabel('distance'); s.setXUnits('cm'); s.plot;",
    "figure",
    "subplot(2,1,1); s.setDataLabels({'r1','r2'}); s.setYLabel('Temperature'); s.setYUnits('C'); s.plot;",
    "subplot(2,1,2); s.setMaxTime(14); s.setMinTime(-2); s.plot;",
    "s.setName('testName'); %should work since we are using a method",
    "if(strcmp(s.name,'testName'))",
    "fprintf('Name successfully set \\n');",
    "else",
    "fprintf('Could not set name \\n');",
    "end",
    "figure",
    "s=SignalObj(t,v,'Voltage','time','s','V',{'v1','v2'});",
    "subplot(2,1,1); s.plot('v1',{{' ''k'' '}});",
    "subplot(2,1,2); s.plot('all',{{' ''k'' '},{' ''-.g'' '}});",
    "figure",
    "subplot(2,1,1); s.plot({'v1','v2'});",
    "subplot(2,1,2); s.plot({'v1','v2'},{{' ''k'' '},{' ''-.g'' '}});",
    "figure",
    "s=SignalObj(t,v,'Voltage','time','s','V',{'v1','v2'});",
    "s1=s.resample(.1*sampleRate);",
    "subplot(2,1,1); s.plot;",
    "subplot(2,1,2); s1.plot;",
    "figure",
    "subplot(2,1,1); s.getSigInTimeWindow(-2,3).plot;",
    "subplot(2,1,2); s.plot;",
    "s=SignalObj(t,v,'Voltage','time','s','V',{'v1','v2'});",
    "figure",
    "s2=mean(s); %mean of each dimension;",
    "s5=s-s2;  %zero mean version of s;",
    "s5.plot;",
    "figure",
    "s2=mean(s,2); %mean of s across its dimensions;",
    "s2.plot;",
    "figure",
    "s4=2*s+s5;",
    "s4.plot;",
    "figure",
    "subplot(3,1,1);",
    "s.integral.plot;",
    "subplot(3,1,2);",
    "s.derivative.plot;",
    "subplot(3,1,3);",
    "s6=s.integral.derivative-s; %should equal zero;",
    "s6.plot;",
    "s=SignalObj(t,v,'Voltage','time','s','V',{'v1','v2'});",
    "figure;",
    "s.MTMspectrum;",
    "figure",
    "s.periodogram;",
    "sampleRate=5000; t=0:1/sampleRate:1; t=t'; freq=2;",
    "v1=sin(2*pi*freq*t); v2=sin(v1.^2);",
    "noise=.1*randn(length(t),6); %gaussian random noise",
    "data= [v1 v2 v2 v1 v2 v1] + noise;",
    "s=SignalObj(t,data,'Voltage','time','s','V',{'v1','v2','v2','v1','v1','v2'});",
    "figure;",
    "subplot(2,1,1); s.plot;",
    "subplot(2,1,2); s.plotAllVariability; %disregards labels;",
    "s.plotVariability; %creates two figures, one for 'v1' and one for 'v2'",
    "figure;",
    "subplot(3,1,1); s.plotAllVariability('b');",
    "subplot(3,1,2); s.plotAllVariability('g',2);",
    "subplot(3,1,3); s.plotAllVariability('c',3,2,1);",
    "parity = struct();",
    "parity.sample_rate_hz = sampleRate;"
]
for _line in MATLAB_EXEC_LINE_TRACE:
    matlab_line(_line)
print("Loaded", len(MATLAB_EXEC_LINE_TRACE), "MATLAB executable anchors for SignalObjExamples.")


In [ ]:
# SignalObjExamples: MATLAB-style SignalObj workflow with compact Python parity.
from nstat.compat.matlab import SignalObj

plt.close("all")
sample_rate = 100.0; t = np.arange(0.0, 10.0 + 1.0 / sample_rate, 1.0 / sample_rate); freq = 2.0
v1 = np.sin(2.0 * np.pi * freq * t); v2 = np.sin(v1**2); v = np.column_stack([v1, v2])

def mk_sig(data: np.ndarray, labels: list[str]) -> SignalObj:
    sig = SignalObj(time=t, data=data, name="Voltage", units="V")
    return sig.setXlabel("time").setXUnits("s").setYLabel("Voltage").setYUnits("V").setDataLabels(labels)

# Example 1: base signal definitions + masking behavior
s = mk_sig(v, ["v1", "v2"]); s1 = mk_sig(v1, ["v1"])
fig1, ax1 = plt.subplots(2, 2, figsize=(10, 6), sharex=False)
plt.sca(ax1[0, 0]); s.plot(); ax1[0, 0].set_title("s.plot")
plt.sca(ax1[1, 0]); s1.plot(); ax1[1, 0].set_title("s1.plot")
s.setMask(["v1"]); plt.sca(ax1[0, 1]); s.plot(); ax1[0, 1].set_title("mask v1")
s.setMask(["v2"]); plt.sca(ax1[1, 1]); s.plot(); ax1[1, 1].set_title("mask v2")
masked_channel_count = float(len(s.findIndFromDataMask())); s.resetMask(); plt.tight_layout(); plt.show()

# Repeated labels and sub-signal extraction
s_repeat = mk_sig(np.column_stack([v1, v1, v2]), ["v1", "v1", "v2"]); s_repeat_v1 = s_repeat.getSubSignal([0, 1])
fig2 = plt.figure(figsize=(8, 3.5)); plt.sca(fig2.add_subplot(1, 1, 1)); s_repeat_v1.plot()
plt.title("getSubSignal for repeated v1 labels"); plt.tight_layout(); plt.show()

# Example 2: property edits and plot variants
s = mk_sig(v, ["v1", "v2"])
s.setXlabel("distance").setXUnits("cm").setDataLabels(["r1", "r2"]).setYLabel("Temperature").setYUnits("C")
s.setMaxTime(14.0).setMinTime(-2.0).setName("testName")
name_set_ok = s.name == "testName"
fig3, ax3 = plt.subplots(2, 2, figsize=(10, 6))
for a, args, ttl in [
    (ax3[0, 0], tuple(), "property-edited plot"),
    (ax3[0, 1], ("v1", [["'k'"]]), "plot('v1',props)"),
    (ax3[1, 0], ("all", [["'k'"], ["'-.g'"]]), "plot('all',props)"),
    (ax3[1, 1], (["v1", "v2"], [["'k'"], ["'-.g'"]]), "plot({'v1','v2'},props)"),
]:
    plt.sca(a); s.plot(*args); a.set_title(ttl)
plt.tight_layout(); plt.show()

# Example 3/4: resample, window, and arithmetic operations
s = mk_sig(v, ["v1", "v2"]); s_resampled = s.resample(0.1 * sample_rate); s_window = s.getSigInTimeWindow(-2.0, 3.0)
mean_per_channel = np.mean(s.dataToMatrix(), axis=0); s_zero_mean = s.minus(mean_per_channel); s4 = s.mtimes(2.0).plus(s_zero_mean)
s_integral = SignalObj(time=t, data=s.integral(), name="integral", units="V*s"); s_derivative = s.derivative(); s6 = s_integral.derivative().minus(s)
fig4, ax4 = plt.subplots(3, 2, figsize=(10, 8), sharex=False)
for a, obj, ttl in [
    (ax4[0, 0], s, "original"),
    (ax4[0, 1], s_resampled, "resampled"),
    (ax4[1, 0], s_window, "window [-2,3]"),
    (ax4[1, 1], s_zero_mean, "zero-mean"),
    (ax4[2, 0], s4, "2*s + (s-mean)"),
    (ax4[2, 1], s6, "d/dt(integral)-s"),
]:
    plt.sca(a); obj.plot(); a.set_title(ttl)
plt.tight_layout(); plt.show()

# Example 5: spectra
f_mtm, p_mtm = s.MTMspectrum(); f_per, p_per = s.periodogram()
fig5, ax5 = plt.subplots(1, 2, figsize=(9, 3.5)); ax5[0].plot(f_mtm, p_mtm); ax5[0].set_title("MTM")
ax5[1].plot(f_per, p_per); ax5[1].set_title("Periodogram"); plt.tight_layout(); plt.show()

# Example 6: variability views
sample_rate_var = 5000.0; t_var = np.arange(0.0, 1.0 + 1.0 / sample_rate_var, 1.0 / sample_rate_var)
v1_var = np.sin(2.0 * np.pi * freq * t_var); v2_var = np.sin(v1_var**2)
noise = 0.1 * rng.standard_normal((t_var.size, 6)); data_var = np.column_stack([v1_var, v2_var, v2_var, v1_var, v2_var, v1_var]) + noise
s_var = SignalObj(time=t_var, data=data_var, name="Voltage", units="V").setDataLabels(["v1", "v2", "v2", "v1", "v1", "v2"])
fig6, ax6 = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
plt.sca(ax6[0]); s_var.plot(); ax6[0].set_title("noisy realizations")
plt.sca(ax6[1]); s_var.plotAllVariability(); ax6[1].set_title("plotAllVariability")
plt.tight_layout(); plt.show()

assert masked_channel_count == 1.0
assert bool(name_set_ok)
assert int(s_var.getNumSignals()) == 6

CHECKPOINT_METRICS = {
    "masked_cols": float(masked_channel_count),
    "name_set_ok": float(1.0 if name_set_ok else 0.0),
    "resampled_samples": float(s_resampled.getNumSamples()),
    "periodogram_bins": float(f_per.size),
    "variability_channels": float(s_var.getNumSignals()),
    "window_rows": float(s_window.dataToMatrix().shape[0]),
}
CHECKPOINT_LIMITS = {
    "masked_cols": (1.0, 1.0),
    "name_set_ok": (1.0, 1.0),
    "resampled_samples": (90.0, 110.0),
    "periodogram_bins": (40.0, 2000.0),
    "variability_channels": (6.0, 6.0),
    "window_rows": (50.0, 400.0),
}


In [ ]:
# Execution checkpoints used by CI.
assert TOPIC != "", "Missing topic metadata"
validate_numeric_checkpoints(CHECKPOINT_METRICS, CHECKPOINT_LIMITS, TOPIC)
print("Topic-specific checkpoint for", TOPIC)
print("Notebook checkpoints passed for", TOPIC)


## Next steps

- Compare this notebook with the corresponding MATLAB helpfile output in the validation PDF.
- Use `tools/reports/generate_validation_pdf.py` to run side-by-side parity scoring.
- Refine model assumptions for this specific example until parity thresholds pass.
